# RedLamp MLP Baseline - Gradient Conflict Profiling (Demo)

In [ ]:
import torch
from src.data.augment import REDLAMP_MULTICLASS_CLASS_NAMES
from src.models.redlamp_mlp_baseline import RedLampMLPBaseline

In [ ]:
model = RedLampMLPBaseline(
    input_dim=1,
    window_size=20,
    latent_dim=64,
    mlp_num_linear_layers=3,
    classifier_dim=16,
    num_classes=len(REDLAMP_MULTICLASS_CLASS_NAMES),
    enable_gradient_conflict_profiling=True,
    gradient_log_every_n_steps=1,
    gradient_ema_alpha=0.1,
    gradient_sma_window=50,
)

In [ ]:
batch = {
    "x": torch.randn(8, 20, 1),
    "point_labels": torch.zeros(8, 20, dtype=torch.long),
    "mask": None,
    "timestamps": None,
    "meta": [{"entity_id": "demo"} for _ in range(8)],
}

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)
step_output = model.training_step(batch)
step_output["loss"].backward()
optimizer.step()

In [ ]:
focus_keys = [k for k in step_output["log"] if "/focus/" in k]
for k in sorted(focus_keys):
    print(k, ":", step_output["log"][k])